# Advanced Machine Learning Final Project
- Author: César Núñez
- Date: March 10th 2026

## Data preparation

This section performs the data preparation before implementing any downstream NLP tasks. It uses a dataset that was scraped using the `social_conflicts_peru.scrapers` module whose main source are PDF documents obtained from the National Ombudsman of Peru [repository](https://www.defensoria.gob.pe/categorias_de_documentos/reportes/). The result is a structured dataset that can be used consistently for both conflict-type and dialogue classification.

This notebook is part of one personal project about social conflicts in Peru, which is available [here](https://github.com/cesarnunezh/social-conflicts-peru/blob/main/).

To avoid running the `scrapers` module again, I'm using a sample of conflict ocurrances reported from April 2021 to January 2026, available in this [json file](https://github.com/cesarnunezh/social-conflicts-peru/blob/main/nlp_analysis/conflict_occurrences_active_latent.json).

In [1]:
# Required libraries
%load_ext autoreload
%autoreload 2

import json, nltk
from nltk.corpus import stopwords
import polars as pl
from social_conflicts_peru.config import directories
from nltk.tokenize import word_tokenize

In [2]:
# Importing data
with open(directories.PROCESSED_DATA / "conflict_occurrences_active_latent.json" , "r") as file:
    data = pl.DataFrame(json.load(file))

In [3]:
# Renaming columns
data.columns = ['conflict_type', 'date_init', 'conflict_text', 'location', 'actors_1', 'actors_2', 'actors_3', 'event_text', 
                'dialogo', '_page_start', '_page_end', '_confidence', '_section', '_subsection', '_source_pdf', '_seccion', 
                '_grupo_conflicto', '_case_id_local', 'conflict_uid', 'state', 'report_number']

In [4]:
# Creating variables
data = data.filter(pl.col("dialogo").is_in(["HAY DIÁLOGO", "NO HAY DIÁLOGO"]))
data = data.filter(pl.len().over("conflict_uid") > 5).sort(["conflict_uid", "report_number"])

In [5]:
# Cleaning types
data = data.with_columns(
    pl.when(
        pl.col("conflict_type").str.contains(
            r"(?i)^por asuntos? de gobierno local|^asuntos? de gobierno local"
        )
    )
    .then(pl.lit("Asuntos de gobierno local"))

    .when(
        pl.col("conflict_type").str.contains(
            r"(?i)^por asuntos? de gobierno regional|^asuntos? de gobiern(os)? regional|^asuntos? de gobiernos regional|^asuntos? de gobierno regional"
        )
    )
    .then(pl.lit("Asuntos de gobierno regional"))

    .when(
        pl.col("conflict_type").str.contains(
            r"(?i)^asuntos? de gobierno nacional"
        )
    )
    .then(pl.lit("Asuntos de gobierno nacional"))

    .when(
        pl.col("conflict_type").str.contains(
            r"(?i)^cultivo ilegal de hoja de coca"
        )
    )
    .then(pl.lit("Cultivo ilegal de hoja de coca"))

    .when(
        pl.col("conflict_type").str.contains(
            r"(?i)^demarcación territorial"
        )
    )
    .then(pl.lit("Demarcación territorial"))

    .when(
        pl.col("conflict_type").str.contains(
            r"(?i)^socioambiental"
        )
    )
    .then(pl.lit("Socioambiental"))

    .otherwise(pl.col("conflict_type"))
    .alias("conflict_type")
)

In [6]:
# Cleaning date column
month_map = {
    "enero": "01",
    "febrero": "02",
    "marzo": "03",
    "abril": "04",
    "mayo": "05",
    "junio": "06",
    "julio": "07",
    "agosto": "08",
    "septiembre": "09",
    "setiembre": "09",
    "octubre": "10",
    "noviembre": "11",
    "diciembre": "12",
}

month_pattern = "|".join(month_map.keys())

data = (
    data
    .with_columns(
        pl.col("date_init").str.strip_chars().alias("date_init")
    )
    .with_columns(
        pl.col("date_init")
        .str.extract(rf"(?i)^({month_pattern})", 1)
        .str.to_lowercase()
        .replace(month_map)
        .alias("month_num"),

        pl.col("date_init")
        .str.extract(r"(?i)(\d{4})", 1)
        .alias("year"),

        pl.col("date_init")
        .str.extract(
            rf"(?i)^(?:{month_pattern})(?:\s+de)?[,\s]+(?:\d{{4}})\.?,?\s*(.*)$",
            1
        )
        .alias("rest_text"),
    )
    .with_columns(
        pl.when(
            pl.col("year").is_not_null() & pl.col("month_num").is_not_null()
        )
        .then(
            pl.concat_str([
                pl.col("year"),
                pl.lit("-"),
                pl.col("month_num"),
                pl.lit("-01"),
            ]).str.strptime(pl.Date, "%Y-%m-%d")
        )
        .otherwise(None)
        .alias("clean_date"),

        pl.when(pl.col("rest_text") == "")
        .then(None)
        .otherwise(pl.col("rest_text"))
        .alias("rest_text"),
    )
    .drop(["month_num", "year"])
)

In [7]:
# Adding text found on dates to event text.
data = data.with_columns(
    pl.col("event_text").fill_null(pl.col("rest_text")).alias("event_text")
)

In [8]:
# Removing events without information and 
data = data.filter(
    pl.col('event_text') != 'se registraron nuevos hechos durante el mes.',
    pl.col('event_text') != 'se registraron nuevos hechos durante el mes..',
)

In [9]:
# Lowering case of conflict_type
data = data.with_columns(
    pl.col('conflict_type').str.to_lowercase().alias('conflict_type')
)

In [10]:
# Selecting useful columns
df_final = data[:, ["conflict_type", "date_init", "conflict_text", "location", "actors_1", "actors_2", "actors_3", "event_text", "dialogo", "conflict_uid", "report_number", "clean_date"]]

In [11]:
df_final["conflict_type"].value_counts()

conflict_type,count
str,u32
"""otros asuntos""",35
"""socioambiental""",2029
"""cultivo ilegal de hoja de coca""",2
"""demarcación territorial""",52
"""asuntos de gobierno nacional""",300
"""asuntos de gobierno local""",93
"""laboral""",61
"""comunal""",206
"""asuntos de gobierno regional""",171


In [12]:
df_final["dialogo"].value_counts()

dialogo,count
str,u32
"""HAY DIÁLOGO""",2186
"""NO HAY DIÁLOGO""",763


### Feature extraction

This subsection defines the variables used in the classification tasks. In particular, I transform the target variable related to
dialogue into a dummy variable (0 or 1), preserve the conflict identifiers needed for grouped splitting, and keep the textual fields that
will later be used as model inputs at both the conflict level and the event level.

In [13]:
# Dialogo to dummy variable
df_final = df_final.with_columns(
    pl.col("dialogo")
    .replace({
        "HAY DIÁLOGO": 1,
        "NO HAY DIÁLOGO": 0
    })
    .cast(pl.Int8)
    .alias("dialogo")
)

In [14]:
df_final = df_final.unique(
    subset=["conflict_uid", "report_number", "event_text"],
    keep="first"
)

### Class imbalance

Before training the models, I inspect the class distribution to identify possible imbalance problems. This step is important because
skewed labels can bias both traditional and transformer-based classifiers toward the majority class, so the results must later be
interpreted together with macro-level metrics and class-sensitive evaluation.

In [15]:
pl.DataFrame({
    "total_rows": [df_final.height],
    "positive": [df_final.filter(pl.col("dialogo") == 1).height],
    "negative": [df_final.filter(pl.col("dialogo") == 0).height],
}).with_columns(
    (pl.col("positive") / pl.col("total_rows")).alias("positive_ratio"),
    (pl.col("negative") / pl.col("total_rows")).alias("negative_ratio")
)

total_rows,positive,negative,positive_ratio,negative_ratio
i64,i64,i64,f64,f64
2947,2184,763,0.741093,0.258907


### Text cleaning

This subsection applies basic text normalization to the conflict and event descriptions. The goal is to reduce noise and make the
textual inputs more consistent across observations, while preserving the core semantic information needed for classification.

In [16]:
nltk.download("stopwords")
nltk.download("punkt")
nltk.download("punkt_tab")
SPA_STOP = stopwords.words("spanish")

def clean_text(text: str, stopwords: list[str] = SPA_STOP) -> str:

    words = word_tokenize(text.lower())
    clean_words = [word for word in words if word not in stopwords]
    return " ".join(clean_words)

[nltk_data] Downloading package stopwords to /home/canun/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package punkt to /home/canun/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to /home/canun/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


In [17]:
df_final = df_final.with_columns(
    pl.col('conflict_text')
    .map_elements(clean_text, return_dtype=pl.String)
    .alias('clean_conflict_text'),
    pl.col('event_text')
    .map_elements(clean_text, return_dtype=pl.String)
    .alias('clean_event_text')
)

In [18]:
df_final = df_final[:, ["dialogo", "conflict_type", "conflict_uid", "report_number", "clean_date", "clean_conflict_text", "clean_event_text", "date_init", "location", "actors_1", "actors_2", "actors_3"]]

## Fine-Tunning DISTILBERT and BERTO (spanish BERT) classification models

This section fine-tunes transformer-based classifiers for two related but distinct tasks. The first model predicts the type of conflict
using conflict-level descriptions, while the second predicts whether a specific event shows evidence of dialogue using event-level text.

1. `conflict_type` multiclass model using conflict-level rows (`clean_conflict_text`, `location`, `actors_1`, `actors_2`, `actors_3`, `date_init`).
2. `dialogo` binary model using event-level rows (`clean_event_text`) with group split by `conflict_uid`.

Both pipelines follow a comparable training and evaluation procedure in order to support a fair comparison across models.

In [21]:
from dataclasses import dataclass
from typing import Dict

import numpy as np
import polars as pl
import torch
from datasets import Dataset, DatasetDict
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
    precision_recall_fscore_support,
    average_precision_score,
)
from sklearn.model_selection import train_test_split, GroupShuffleSplit
from sklearn.utils.class_weight import compute_class_weight
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    DataCollatorWithPadding,
    EarlyStoppingCallback,
    TrainingArguments,
    Trainer,
)

In [22]:
SEED = 42

@dataclass
class LabelMapping:
    label2id: Dict[str, int]
    id2label: Dict[int, str]


def build_label_mapping(values: list[str]) -> LabelMapping:
    labels = sorted(set(values))
    label2id = {label: ix for ix, label in enumerate(labels)}
    id2label = {ix: label for label, ix in label2id.items()}
    return LabelMapping(label2id=label2id, id2label=id2label)


def compute_metrics_factory(num_labels: int):
    def compute_metrics(eval_pred):
        logits, labels = eval_pred
        preds = np.argmax(logits, axis=-1)

        precision_w, recall_w, f1_w, _ = precision_recall_fscore_support(
            labels, preds, average="weighted", zero_division=0
        )

        metrics = {
            "accuracy": accuracy_score(labels, preds),
            "balanced_accuracy": balanced_accuracy_score(labels, preds),
            "precision_weighted": precision_w,
            "recall_weighted": recall_w,
            "f1_weighted": f1_w,
            "f1_macro": f1_score(labels, preds, average="macro", zero_division=0),
        }

        if num_labels == 2:
            probs_pos = torch.softmax(torch.tensor(logits), dim=-1)[:, 1].numpy()
            metrics["pr_auc_positive"] = average_precision_score(labels, probs_pos)

        return metrics

    return compute_metrics


class WeightedTrainer(Trainer):
    def __init__(self, class_weights: torch.Tensor, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self.class_weights = class_weights

    def compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch=None):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        logits = outputs.logits
        loss_fct = torch.nn.CrossEntropyLoss(weight=self.class_weights.to(logits.device))
        loss = loss_fct(logits, labels)
        return (loss, outputs) if return_outputs else loss


def train_text_classifier(
    train_df: pl.DataFrame,
    val_df: pl.DataFrame,
    test_df: pl.DataFrame,
    text_col: str,
    label_col: str,
    model_name: str,
    output_dir: str,
    hf_repo: str,
    num_epochs: int = 6,
    max_length: int = 256,
    train_bs: int = 16,
    eval_bs: int = 32, 
):
    train_df = train_df.select([text_col, label_col]).drop_nulls()
    val_df = val_df.select([text_col, label_col]).drop_nulls()
    test_df = test_df.select([text_col, label_col]).drop_nulls()

    mapping = build_label_mapping(train_df[label_col].cast(pl.String).to_list())

    def encode(df: pl.DataFrame) -> pl.DataFrame:
        return df.with_columns(
            pl.col(label_col)
            .cast(pl.String)
            .replace_strict(mapping.label2id)
            .cast(pl.Int64)
            .alias("label_id")
        )

    train_enc, val_enc, test_enc = encode(train_df), encode(val_df), encode(test_df)

    ds = DatasetDict({
        "train": Dataset.from_polars(train_enc.select([text_col, "label_id"])),
        "validation": Dataset.from_polars(val_enc.select([text_col, "label_id"])),
        "test": Dataset.from_polars(test_enc.select([text_col, "label_id"])),
    })

    tokenizer = AutoTokenizer.from_pretrained(model_name)

    def preprocess(examples):
        out = tokenizer(examples[text_col], truncation=True, max_length=max_length)
        out["labels"] = examples["label_id"]
        return out

    ds_tok = ds.map(preprocess, batched=True)

    y_train = np.array(train_enc["label_id"].to_list())
    class_weights = compute_class_weight(
        class_weight="balanced",
        classes=np.unique(y_train),
        y=y_train,
    )
    class_weights = torch.tensor(class_weights, dtype=torch.float)

    model = AutoModelForSequenceClassification.from_pretrained(
        model_name,
        num_labels=len(mapping.id2label),
        id2label=mapping.id2label,
        label2id=mapping.label2id,
    )

    training_args = TrainingArguments(
        output_dir=output_dir,
        learning_rate=2e-5,
        per_device_train_batch_size=train_bs,
        per_device_eval_batch_size=eval_bs,
        num_train_epochs=num_epochs,
        weight_decay=0.01,
        eval_strategy="epoch",
        save_strategy="best",
        load_best_model_at_end=True,
        metric_for_best_model="f1_macro",
        greater_is_better=True,
        logging_strategy="epoch",
        push_to_hub=True,
        hub_model_id=hf_repo,
        seed=SEED,
        disable_tqdm=True,
        report_to="none",
    )

    trainer = WeightedTrainer(
        class_weights=class_weights,
        model=model,
        args=training_args,
        train_dataset=ds_tok["train"],
        eval_dataset=ds_tok["validation"],
        processing_class=tokenizer,
        data_collator=DataCollatorWithPadding(tokenizer=tokenizer),
        compute_metrics=compute_metrics_factory(num_labels=len(mapping.id2label)),
        callbacks=[EarlyStoppingCallback(early_stopping_patience=2)],
    )

    train_output = trainer.train()
    val_metrics = trainer.evaluate(ds_tok["validation"], metric_key_prefix="val")
    test_metrics = trainer.evaluate(ds_tok["test"], metric_key_prefix="test")

    return {
        "trainer": trainer,
        "tokenizer": tokenizer,
        "mapping": mapping,
        "train_output": train_output,
        "val_metrics": val_metrics,
        "test_metrics": test_metrics,
    }


### A) Conflict Type Classifier (Multiclass)

This model treats conflict type as a multiclass classification problem. Each observation represents a conflict-level description, and
the objective is to assign it to the correct category, such as socio-environmental, communal, or government-related conflict. This task
evaluates whether compact transformer models can recover meaningful thematic differences from the text alone.

In [23]:
from huggingface_hub import login, create_repo
from social_conflicts_peru.config import settings

login(settings.HF_TOKEN)

create_repo(
    repo_id="cesarnunezh/distilbert-conflict-classifier",
    private=True,
    exist_ok= True
)

create_repo(
    repo_id="cesarnunezh/berto-conflict-classifier",
    private=True,
    exist_ok= True
)

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


RepoUrl('https://huggingface.co/cesarnunezh/berto-conflict-classifier', endpoint='https://huggingface.co', repo_type='model', repo_id='cesarnunezh/berto-conflict-classifier')

In [24]:
# Conflict-level dataset: one row per conflict_uid
conflict_df = (
    df_final
    .select(["conflict_uid", "conflict_type", "clean_conflict_text", "location", "date_init", "actors_1", "actors_2", "actors_3"])
    .drop_nulls()
    .unique(subset=["conflict_uid"], keep="first")
    .with_columns(
        pl.format("[CONFLICTO] {} [UBICACION] {} [FECHA] {} [ACTORES 1] {} [ACTORES 2] {} [ACTORES 3] {}", 
                  pl.col("clean_conflict_text"), pl.col("location"), pl.col("date_init"), pl.col("actors_1"), pl.col("actors_2"), pl.col("actors_3"))
                  .alias("model_text")
    )
)

# Remove very rare labels
min_count = 10
valid_labels = (
    conflict_df["conflict_type"]
    .value_counts()
    .filter(pl.col("count") >= min_count)["conflict_type"]
    .to_list()
)
conflict_df = conflict_df.filter(pl.col("conflict_type").is_in(valid_labels))

train_conflict, temp_conflict = train_test_split(
    conflict_df.to_pandas(),
    test_size=0.30,
    stratify=conflict_df["conflict_type"].to_pandas(),
    random_state=SEED,
)

val_conflict, test_conflict = train_test_split(
    temp_conflict,
    test_size=0.50,
    stratify=temp_conflict["conflict_type"],
    random_state=SEED,
)

train_conflict = pl.from_pandas(train_conflict)
val_conflict = pl.from_pandas(val_conflict)
test_conflict = pl.from_pandas(test_conflict)

print("Conflict split sizes:", train_conflict.height, val_conflict.height, test_conflict.height)
print("Train class counts:")
print(train_conflict["conflict_type"].value_counts().sort("count", descending=True))

Conflict split sizes: 172 37 37
Train class counts:
shape: (5, 2)
┌──────────────────────────────┬───────┐
│ conflict_type                ┆ count │
│ ---                          ┆ ---   │
│ str                          ┆ u32   │
╞══════════════════════════════╪═══════╡
│ socioambiental               ┆ 111   │
│ asuntos de gobierno nacional ┆ 23    │
│ comunal                      ┆ 16    │
│ asuntos de gobierno regional ┆ 12    │
│ asuntos de gobierno local    ┆ 10    │
└──────────────────────────────┴───────┘


In [ ]:
# CONFLICT_MODEL = "distilbert/distilbert-base-multilingual-cased"

# conflict_run = train_text_classifier(
#     train_df=train_conflict,
#     val_df=val_conflict,
#     test_df=test_conflict,
#     text_col="model_text",
#     label_col="conflict_type",
#     model_name=CONFLICT_MODEL,
#     output_dir=str(directories.ROOT_DIR / "nlp_analysis/models_conflict_type"),
#     hf_repo="cesarnunezh/distilbert-conflict-classifier",
#     num_epochs=10,
#     max_length=256,
# )

# print("Validation metrics:", conflict_run["val_metrics"])
# print("Test metrics:", conflict_run["test_metrics"])


In [ ]:
# CONFLICT_MODEL = "dccuchile/bert-base-spanish-wwm-cased"

# conflict_run = train_text_classifier(
#     train_df=train_conflict,
#     val_df=val_conflict,
#     test_df=test_conflict,
#     text_col="model_text",
#     label_col="conflict_type",
#     model_name=CONFLICT_MODEL,
#     output_dir=str(directories.ROOT_DIR / "nlp_analysis/models_conflict_type"),
#     hf_repo="cesarnunezh/berto-conflict-classifier",
#     num_epochs=10,
#     max_length=256,
# )

# print("Validation metrics:", conflict_run["val_metrics"])
# print("Test metrics:", conflict_run["test_metrics"])


### B) Dialogo vs No Dialogo Classifier (Binary, Group Split by Conflict)

This model frames dialogue detection as a binary classification task at the event level. Each observation corresponds to a reported
event, and the target indicates whether the text provides evidence of dialogue between the parties. To reduce leakage, the train,
validation, and test splits are grouped by conflict identifier so that events from the same conflict do not appear across multiple splits.

In [25]:
create_repo(
    repo_id="cesarnunezh/distilbert-event-classifier",
    private=True,
    exist_ok= True
)

create_repo(
    repo_id="cesarnunezh/berto-event-classifier",
    private=True,
    exist_ok= True
)

RepoUrl('https://huggingface.co/cesarnunezh/berto-event-classifier', endpoint='https://huggingface.co', repo_type='model', repo_id='cesarnunezh/berto-event-classifier')

In [26]:
# Event-level dataset for dialogo classification
dialogo_df = (
    df_final
    .select(["conflict_uid", "dialogo", "clean_event_text", "clean_conflict_text"])
    .drop_nulls()
    .with_columns(
        pl.format("[CONFLICTO] {} [EVENTO] {}", pl.col("clean_conflict_text"), pl.col("clean_event_text")).alias("model_text")
    )
)

# Group split by conflict_uid to avoid leakage across train/val/test
all_groups = dialogo_df["conflict_uid"].to_pandas()

gss_1 = GroupShuffleSplit(n_splits=1, test_size=0.15, random_state=SEED)
train_val_idx, test_idx = next(gss_1.split(dialogo_df.to_pandas(), groups=all_groups))

train_val_df = dialogo_df[train_val_idx]
test_df = dialogo_df[test_idx]

gss_2 = GroupShuffleSplit(n_splits=1, test_size=0.1765, random_state=SEED)  # ~15% of total
train_idx, val_idx = next(gss_2.split(train_val_df.to_pandas(), groups=train_val_df["conflict_uid"].to_pandas()))

train_df = train_val_df[train_idx]
val_df = train_val_df[val_idx]

print("Dialogo split sizes:", train_df.height, val_df.height, test_df.height)
print("Train dialogo ratio:")
print(train_df["dialogo"].value_counts().sort("dialogo"))
print("Unique conflicts by split:")
print({
    "train": train_df["conflict_uid"].n_unique(),
    "val": val_df["conflict_uid"].n_unique(),
    "test": test_df["conflict_uid"].n_unique(),
})

Dialogo split sizes: 2030 557 360
Train dialogo ratio:
shape: (2, 2)
┌─────────┬───────┐
│ dialogo ┆ count │
│ ---     ┆ ---   │
│ i8      ┆ u32   │
╞═════════╪═══════╡
│ 0       ┆ 550   │
│ 1       ┆ 1480  │
└─────────┴───────┘
Unique conflicts by split:
{'train': 200, 'val': 43, 'test': 43}


In [27]:
# Keep labels as strings for a stable, explicit mapping
train_df = train_df.with_columns(pl.when(pl.col("dialogo") == 1).then(pl.lit("HAY_DIALOGO")).otherwise(pl.lit("NO_DIALOGO")).alias("dialogo_label"))
val_df = val_df.with_columns(pl.when(pl.col("dialogo") == 1).then(pl.lit("HAY_DIALOGO")).otherwise(pl.lit("NO_DIALOGO")).alias("dialogo_label"))
test_df = test_df.with_columns(pl.when(pl.col("dialogo") == 1).then(pl.lit("HAY_DIALOGO")).otherwise(pl.lit("NO_DIALOGO")).alias("dialogo_label"))

# DIALOGO_MODEL = "distilbert/distilbert-base-multilingual-cased"

# dialogo_run = train_text_classifier(
#     train_df=train_df,
#     val_df=val_df,
#     test_df=test_df,
#     text_col="model_text",
#     label_col="dialogo_label",
#     model_name=DIALOGO_MODEL,
#     output_dir=str(directories.ROOT_DIR / "nlp_analysis/models_dialogo"),
#     hf_repo="cesarnunezh/distilbert-event-classifier",
#     num_epochs=6,
#     max_length=256,
# )

# print("Validation metrics:", dialogo_run["val_metrics"])
# print("Test metrics:", dialogo_run["test_metrics"])

In [28]:
# DIALOGO_MODEL = "dccuchile/bert-base-spanish-wwm-cased"

# dialogo_run = train_text_classifier(
#     train_df=train_df,
#     val_df=val_df,
#     test_df=test_df,
#     text_col="model_text",
#     label_col="dialogo_label",
#     model_name=DIALOGO_MODEL,
#     output_dir=str(directories.ROOT_DIR / "nlp_analysis/models_dialogo"),
#     hf_repo="cesarnunezh/berto-event-classifier",
#     num_epochs=6,
#     max_length=256,
# )

# print("Validation metrics:", dialogo_run["val_metrics"])
# print("Test metrics:", dialogo_run["test_metrics"])


### Model evaluation

This subsection compares model performance using metrics that are appropriate for potentially imbalanced classification settings. Rather
performs well and where it still fails.

In [28]:
from transformers import pipeline

DISTIL_MODEL = "distilbert/distilbert-base-multilingual-cased"
BERTO_MODEL = "dccuchile/bert-base-spanish-wwm-cased"

distilbert_conflict_classifier = pipeline(
    "text-classification",
    model="cesarnunezh/distilbert-conflict-classifier",
    tokenizer=AutoTokenizer.from_pretrained(DISTIL_MODEL),
    truncation=True,
    max_length=256,
)

distilbert_dialogo_classifier = pipeline(
    "text-classification",
    model="cesarnunezh/distilbert-event-classifier",
    tokenizer= AutoTokenizer.from_pretrained(DISTIL_MODEL),
    truncation=True,
    max_length=256,
)

berto_conflict_classifier = pipeline(
    "text-classification",
    model="cesarnunezh/berto-conflict-classifier",
    tokenizer=AutoTokenizer.from_pretrained(BERTO_MODEL),
    truncation=True,
    max_length=256,
)

berto_dialogo_classifier = pipeline(
    "text-classification",
    model="cesarnunezh/berto-event-classifier",
    tokenizer= AutoTokenizer.from_pretrained(BERTO_MODEL),
    truncation=True,
    max_length=256,
)

Loading weights: 100%|██████████| 201/201 [00:00<00:00, 1585.83it/s]


In [29]:
# Sample cases from 2018
# https://www.defensoria.gob.pe/wp-content/uploads/2019/01/Conflictos-Sociales-N%C2%B0-178-Diciembre-2018.pdf

sample_conflicts = [
    {'label' : "HAY_DIALOGO",
     'conflict_type'  : "asuntos de gobierno regional",
     'date_init' : """Diciembre de 2018.""",
     'location' : """Distrito de Huaraz, provincia de Huaraz, región
Áncash.""",
     'actors_1' : 'CISEA Nicrupamapa, I.E.I. Nicrupampa.',
     'actors_2' : """Dirección Regional de Salud de Ancash
(DIRESA), Dirección Regional de Educación.""",
     'actors_3' : """Defensoría del Pueblo, Secretaría de Gestión
Social y Diálogo de la Presidencia del Consejo de Ministros.""",
     'conflict_text' : """El Centro Integrado de Salud, Educación y Agricultura
(CISEA) de Nicrupampa pretende recuperar un terreno de 200
metros para la construcción de un centro materno infantil, que
está actualmente ocupado por la Institución Educativa Inicial de
Nicrupampa""",
     'event_text' : """El 18 de diciembre, se realizó una reunión en la Institución
Educativa con representantes del Dirección Regional de Salud y
El 18 de diciembre, se reunieron en la Institución Educativa
Inicial de Nicrupampa (I.E.I.) representantes de la Procuraduría
del Gobierno Regional de Áncash, el Director de la Red de Salud
de Huaylas Sur y la DIrectora de la Institución Educativa para
tratar el problema del terreno en cuestión. El representante de la
pPocuraduría del Gobierno Regional de Áncash señaló que
podrían realizar un desalojo administrativo. Por su parte, la
directora de la Institución Educativa mencionó que tenía un acta
de donación otorgada por el ex gobernador regional de Áncash.
El 19 de diciembre, mediante el Oficio N°1313-2018-DP/ODANC, la Defensoría del Pueblo hizo traslado al Director de la
Red de Salud de Huaylas Sur de la queja interpuesta por una
ciudadana contra el Centro de Salud de Nicrupampa. Personal
de este centro de salud le informó que no iba atender al público
porque saldrían a realizar una movilización. Asimismo, señaló
que tampoco atendían a las personas que estaban en la cola
desde temprano.
El 3 de enero, la Defensoría se reunió con el Jefe de la Unidad
del Personal del Centro de Salud de Nicrupampa, quien
manifestó que no existe una disposición que autorice la
suspensión de las labores del personal de salud y que
procederán a la constatación de la atención al público. Además,
se comprometió a restablecer la atención a los ciudadanos.
Asimismo, el 7 de enero el Jefe de la Unidad de Personal y al
Asesor Legal de la Red de Salud de Nicrupampa informaron a la
Defensoría del Pueblo que agilizarían los trámites para retomar
las actividades del Centro de Salud. 
El 8 de enero, la Defensoría del Pueblo envió al Director
Ejecutivo de Red de Salud el Oficio N°005-2019-DP/OD-ANC,
donde se le recomienda disponer lo siguiente:
- Adoptar medidas correcticas para garantizar la atención de
los pacientes del Centro de Salud Nicrupampa.
- Emitir un pronunciamiento sobre la legalidad de la medida
de protesta adoptada por el Centro de Salud e implementar
las acciones administrativas disciplinarias.
- Adoptar acciones a fin que este tipo de situación no vuelvan
a repetirse y coordinar para optimizar los flujos de atención a los pacientes.
El mismo día, la Defensoría se reunió con la Gerencia de
Desarrollo Social del Gobierno Regional de Áncash para abordar
el problema entre el CISEA de Nicrupampa y la I.E.I. En esta
reunión, también participaron representantes de la Secretaria de
Gestión Social y Diálogo (SGSD) de la PCM, el Ministerio de
Agricultura y Riego y la Directora de la Institución Educativa. Se
acordó una próxima reunión el 15 de enero para abordar el tema
con presencia de todos los interesados y directivos tanto del
sector salud como de educación de la región."""},
    {'label': 'NO_DIALOGO',
     'conflict_type' : 'socioambiental',
     'date_init' : """Octubre de 2018.""",
     'location' : """Distrito de Quillo, provincia de Yungay, región
Áncash.""",
     'actors_1' : """Comunidad campesina Vìrgen del Rosario de
Quillo y empresa COPEMINA. """,
     'actors_2' : """Dirección Regional de Energía y Minas del
Gobierno Regional de Áncash, Dirección Regional de Salud del
Gobierno Regional de Áncash, Fiscalía Especializada en Medio
Ambiente de Huaraz, Centro de Salud del Centro Poblado de
Huacho, Red de Salud Pacífico Sur, Policía Nacional del Perú,
Congregación religiosa Hermanas del Buen Socorro en el Perú.
""",
     'actors_3' : """Defensoría del Pueblo.""",
     'conflict_text' : """La Comunidad Campesina Virgen del Rosario de Quillo
demanda la intervención de la Dirección Regional de Energía y
Minas, Dirección Regional de Salud y de la Autoridad Nacional
del Agua, debido a una presunta afectación a la salud por las
actividades mineras de la empresa COPEMINA, cuyo
campamento minero se encuentra en la parte alta de la cuenca
Sechín, próximo a la fuente de agua que abastece al centro
poblado. Asimismo, denuncian que en el cerro Huancapampa
se estaría realizando minería informal, sin fiscalización de las
autoridades respectivas.""",
     'event_text': """El 9 de enero, la Defensoría del Pueblo se reunió con la
Dirección Regional de Salud del Gobierno Regional, con el
objetivo de supervisar si se procedió a realizar la evaluación
médica de la de los pobladores de la comunidad Virgen del
Rosario, en el distrito de Quillo, tras la denuncia de
contaminación con plomo en la sangre. Se tomó conocimiento
que no se realizaron las evaluaciones, motivo por el cual se
exhortó a priorizar estas acciones a fin de prevenir incurrir en
delitos por la omisión o demoras de actos funcionales.
Cabe informar que en atención a esta problemática, la
Defensoría del Pueblo cursó el Oficio N° 0704-2018-DP/ODANC a la Dirección Regional de Energía y Minas de Ancash. La
referida institución, mediante Oficio N° 1854-2018-GRA/DREM,
dio respuesta que el Consorcio Peruano de Minas S.A.C. expuso
que el 13 de agosto de 2018, ha sufrido la paralización forzada
de sus operaciones en la Concesión Minera El Extraño, por un
periodo indeterminado y que no tienen acceso a las
instalaciones de las operaciones. Asimismo, con Oficio N° 1940-
2018-GRA/DREM, informó con relación a los permisos con los
que cuenta el Consorcio Peruano de Minas S.A.C.
1. Cuenta con la aprobación de la Declaración de Impacto
Ambiental (DIA) del proyecto de explotación minera
U.E.A.COPEMINA, en la Unidad Económica
Administrativa COPEMINA.
2. Cuenta con la autorización de inicio de actividades de
desarrollo, preparación del proyecto explotación en la
UEA COPEMINA en la concesión minera El Extraño.
3. Cuenta con certificado de operación minera periodo 2018.
4. Actualmente tiene una paralización forzada de sus
operaciones en la concesión minera El Extraño.
Por su parte, la Autoridad Nacional del Agua, con Oficio N° 408-
2018-ANA-AAA.HCH-ALACHUARMEY, en atención al pedido
realizado por la Defensoría del Pueblo, con Oficio N° 0804-2018-
DP/OD-ANC, informó lo siguiente:
- Al Centro Poblado menor de Huacho no se le entregó
licencia de uso de agua con fines poblacionales, ni se
cuenta con trámites en curso solicitados por la JASS y/o
Municipalidad.
- A la Empresa Peruana de Minas S.A.C. COPEMINA, no
se le entregó licencia de uso de agua con fines mineros,
estando a la fecha sin iniciar algún trámite y/o solicitud.
La DIRESA Ancash, con Oficio N° 002427-2018-Region Ancash
– DIRES- DESI/DAISCS/PP ENT – M.P. hizo presente el informe
sobre la condición de salud, anemia y desnutrición de los niños y
niñas y población en general del Centro Poblado de Huacho –
Quillo – Yungay. Se precisó que la Red Pacífico Sur se
encuentra levantando información en el campo (Huacho) y está
realizando el muestreo de agua para los análisis
correspondientes.
El 25 de setiembre, la Red de Salud Pacífico Sur, mediante
Oficio N°1706-2018-GRA/DIRESA/RSPS/ODI/USC/ASA/PVICA, alcanzó a la Defensoría del Pueblo los resultados de la
situación actual de la localidad de Huacho, distrito de Quillo,
Provincia de Yungay (Informe Técnico N° 077-
GRA/DIRESA/RSP-S/ODI/USC/ASA/PVICA/LLR), el cual se
arriban a las siguientes conclusiones:
- El agua de la localidad de Huacho cuenta con un sistema
de cloración ( Clorinador automático),
- Hay presencia de cloro pero en baja concentración,
- Se recomienda reactivar el sistema de cloración lo más
pronto posible,
Deberá realizar monitoreo mensual para verificar la carga
microbiana del sector con los análisis microbiológicos que serán
tomados en la localidad."""},
    {'label': 'HAY_DIALOGO',
     'conflict_type' : 'socioambiental',
     'date_init' : """Noviembre de 2011.""",
     'location' : """Provincias de Huari y Recuay, región Áncash.""",
     'actors_1' : """Asociación de Municipalidades de Centros
Poblados (AMUCEPS) de Huari, Compañía Minera Antamina
S.A. (CMA), Nyrstar, comunidad campesina Cátac, Federación
Agraria Departamental de Áncash (FADA).""",
     'actors_2' : """Ministerio de Energía y Minas (MINEM),
Ministerio del Ambiente (MINAM), Ministerio de Inclusión social
(MIDIS), Ministerio de Economía (MEF), Dirección General de
Infraestructura Agraria y Riego (DGIAR) del Ministerio de
Agricultura y Riego (MINAGRI), Programa de Desarrollo
Productivo Agrario Rural (AGRORURAL), Programa
Subsectorial de Irrigaciones (PSI), Ministerio de Salud (MINSA),
Sub Región Conchucos, Municipalidades de Huari, Chavín de
Huántar y San Marcos.""",
     'actors_3' : """Oficina General de Gestión Social del
Ministerio de Energía y Minas (OGGS), Defensoría del Pueblo,
Obispado de Huari, Comisión Episcopal de Acción Social
(CEAS), Secretaria de Gestión Social y Diálogo de la
Presidencia del Consejo de Ministros (SGSD-PCM).
""",
     'conflict_text' : """La Asociación de Municipalidades de Centros Poblados
(AMUCEPS) de Huari en la provincia de Huari denuncia el
incumplimiento de las empresas mineras Antamina S.A. y
Nyrstar de sus compromisos de responsabilidad social y por los
impactos generados en el medio ambiente.""",
     'event_text': """El 12 de diciembre, la Defensoría del Pueblo, sostuvo una
reunión con la SGSD - PCM, MINEM, Compañía Minera
Antamina S.A., con el objetivo de realizar el seguimiento al
caso. Se dio cuenta que el 31 de octubre se realizó una reunión
entre AMUCEPS y Antamina, en la cual AMUCEPS solicitó que
el espacio de diálogo se amplíe a una mesa de desarrollo que
cuente con la participación de gobiernos locales ( provincial y
distrital) para atender proyectos de desarrollo y promover
vigilancia ciudadana; que la empresa informe los detalles de la
ejecución del proyecto de forestación denominado Huari I, y se
gestione una reunión con la nueva Directora Ejecutiva de
AGRORURAL, a fin de exponer los alcances del proyecto de
forestación Huari II.
La empresa informó que la consultoría para la adecuación de
los Centros Poblados tiene los Términos de Referencia para ser
puestos a convocatoria. Ambas partes solicitan que se
convoque a una reunión del espacio de diálogo que cuente con
la participación del MINAGRI, PSI, AGRORURAL Y DGIAR,
para que informen el estado de los proyectos de riego. De igual
manera, se ha visto la necesidad de convocar a la Minera Los
Quenuales para que informen el estado de los dos proyectos de
riego a su cargo.
"""},
    {'label': 'HAY_DIALOGO',
     'conflict_type' : 'comunal',
     'date_init' :"""Abril de 2014.""",
     'location' : """Comunidad campesina Lambrama en el distrito de
Lambrama, provincia de Abancay y comunidad campesina
Curpahuasi en el distrito de Curpahuasi, provincia de Grau,
región Apurímac.
""",
     'actors_1' : """Comunidades de Lambrama y Curpahuasi,
alcaldes de los distritos de Lambrama y Curpahuasi.""",
     'actors_2' : """Gerencia Regional del Gobierno Regional
de Apurímac (GRA), Dirección de Demarcación Territorial del
GRA, Sub Gerencia de Saneamiento Físico Legal de la
Propiedad Rural del GRA, Dirección Regional de Agricultura.""",
     'actors_3' : """Defensoría del Pueblo""",
     'conflict_text' : """ Las comunidades campesinas Lambrama y Curpahuasi
se encuentran en disputa por linderos territoriales. Ambas
insisten en que el sector de Taccata pertenece a su jurisdicción.""",
     'event_text': """El 13 de diciembre, se realizó una reunión en las instalaciones
de la OD de Apurímac. En esta reunión fueron convocados los
representantes de las comunidades de Curpahuasi y Lambrama,
no obstante los integrantes de esta última no asistieron.
Asimismo, se contó con la presencia de representantes del
GORE Apurímac, la Región Policial de Apurímac, FORPRAR y
de la OD de Apurímac.
Finalmente, se acordó que FORPRAP convocará a una reunión
para dar conocer el proceso de titulación que se realizará en la
comunidad de Curpahuasi. En esta reunión, se invitará a las
comunidades colindantes."""},
    {'label': 'NO_DIALOGO',
     'conflict_type' : 'socioambiental',
     'date_init' : """Octubre de 2013.""",
     'location' : """Distritos de Deán Valdivia, Cocachacra y Punta de
Bombón, provincia de Islay, región Arequipa.""",
     'actors_1' : """Autoridades (alcaldes de Islay, Cocachacra,
Punta Bombón y Deán Valdivia), agricultores y pobladores de
los distritos de Cocachacra, Deán Valdivia y Punta Bombón de
la provincia de Islay, Frente de Defensa del Valle de Tambo,
Junta de Usuarios Irrigación Ensenada-Mejía-Mollendo, Junta
de Usuarios del Valle de Tambo, empresa minera Southern
Perú Copper Corporation (SPCC).""",
     'actors_2' : """Pobladores de otros distritos de la
provincia de Islay, Federación Departamental de Trabajadores
de Arequipa (FDTA), Frentes de Defensa Macro Regional,
Partido Político Tierra y Libertad, Ministerio de Agricultura y
Riego (MINAGRI), Autoridad Nacional del Agua (ANA),
Ministerio de Energía y Minas (MINEM) y Ministerio del
Ambiente (MINAM), Organismo de Evaluación y Fiscalización
Ambiental (OEFA), Ministerio del Interior (MININTER) - Policía
Nacional de la Policía (PNP), Fuerzas Armadas (FFAA),
Contraloría General de la República, Poder Judicial, Ministerio
Público.""",
     'actors_3' : """ Secretaría de Gestión Social y Diálogo de la
Presidencia del Consejo de Ministros (SGSD-PCM), Gobierno
Regional de Arequipa (GORE Arequipa), Defensoría del Pueblo.""",
     'conflict_text' : """Agricultores, pobladores y autoridades locales de la
provincia de Islay se oponen al proyecto minero Tía María de la
empresa minera Southern Perú Copper Corporation (SPCC) por
el temor de que se generen impactos negativos al ambiente, y
en consecuencia, se afecte la actividad agrícola en la provincia.
Este caso fue reportado en agosto del 2009 hasta abril de 2011
en que se llega a una solución con la emisión de la Resolución
Directoral N.° 105-2011–MEM-AAM del Ministerio de Energía y
Minas que declara inadmisible el Estudio de Impacto Ambiental
del proyecto minero Tía María presentado por la empresa
minera SPCC.""",
     'event_text': """El 3 de diciembre, representantes de la empresa Southern
realizaron una presentación en la ciudad de Arequipa sobre el
proyecto Tía María, señalando que según la encuesta que se
encargó a Ipsos el 59% de la población de la provincia de Islay
está a favor del desarrollo del proyecto, mientras que el 38%
está en contra."""},
    {'label': 'HAY_DIALOGO',
     'conflict_type' : 'asuntos de gobierno nacional',
     'date_init' : """Mayo de 2017.""",
     'location' : """Provincia de Caylloma, región Arequipa""",
     'actors_1' : """Frente de Defensa de la provincia de
Caylloma, Gobierno Regional de Arequipa, Ministerio de
Energía y Minas, Ministerio del Ambiente, Ministerio de
Transportes y Comunicaciones, Ministerio de Agricultura.
Autoridad Autónoma de Majes (AUTODEMA) .""",
     'actors_2' : """""",
     'actors_3' : """Secretaría de Gestión Social y Diálogo de la
Presidencia del Consejo de Ministros (SGSD-PCM), Defensoría
del Pueblo.""",
     'conflict_text' : """Ciudadanos de Caylloma reclaman al Poder Ejecutivo y
al Gobierno Regional de Arequipa tratar sobre la ejecución del
proyecto Majes Siguas II, la represa de Angostura, el asfaltado
de la vía Vizcachani a Orcopampa y la conformación de un
fondo minero""",
     'event_text': """La reunión programada para el 14 de diciembre, de acuerdo al
Acta de la sesión de las submesas de trabajo “Carretera
Vizcachani - Caylloma y Majes Siguas II” (suscrita el 21.11.2018,
en la sede de la Alcadía Provincial de Caylloma) fue
reprogramada para que se desarrolle con las nuevas
autoridades regional y municipales."""}               
]

##### Conflict type classifier

In [30]:
final_count: dict[str,int] = {}
for conflict in sample_conflicts:

    conflict_type = conflict['conflict_type']
    conflict_text = conflict['conflict_text']
    location = conflict['location']
    date_init = conflict['date_init']
    actors_1 = conflict['actors_1']
    actors_2 = conflict['actors_2']
    actors_3 = conflict['actors_3']

    mode_text = f"""[CONFLICT] {conflict_text} [LOCATION] {location} [DATE] {date_init} [ACTORS 1] {actors_1} [ACTORS 2] {actors_2} [ACTORS 3] {actors_3}"""
    distilbert_result = distilbert_conflict_classifier(mode_text)
    berto_result = berto_conflict_classifier(mode_text)

    print(f"True label: {conflict_type} | Distilbert: {distilbert_result[0]['label']}, Berto: {berto_result[0]['label']}")
    if distilbert_result[0]['label'] == conflict_type:
        final_count['distilbert'] = final_count.get('distilbert', 0) + 1
    if berto_result[0]['label'] == conflict_type:
        final_count['berto'] = final_count.get('berto', 0) + 1

print(f"""
Final results: 
    Distilbert: {final_count.get('distilbert', 0)}/{len(sample_conflicts)}
    Berto: {final_count.get('berto', 0)}/{len(sample_conflicts)}
""")

True label: asuntos de gobierno regional | Distilbert: comunal, Berto: asuntos de gobierno nacional
True label: socioambiental | Distilbert: socioambiental, Berto: socioambiental
True label: socioambiental | Distilbert: socioambiental, Berto: socioambiental
True label: comunal | Distilbert: comunal, Berto: asuntos de gobierno local
True label: socioambiental | Distilbert: comunal, Berto: socioambiental
True label: asuntos de gobierno nacional | Distilbert: asuntos de gobierno regional, Berto: socioambiental

Final results: 
    Distilbert: 3/6
    Berto: 3/6



##### Dialogo classifier

In [31]:
final_count: dict[str,int] = {}
for conflict in sample_conflicts:

    dialogo = conflict['label']
    conflict_text = conflict['conflict_text']
    event_text = conflict['event_text']

    distilbert_result = distilbert_dialogo_classifier(f"[CONFLICTO] {conflict_text} [EVENTO] {event_text}")
    berto_result = berto_dialogo_classifier(f"[CONFLICTO] {conflict_text} [EVENTO] {event_text}")

    print(f"True label: {dialogo} | Distilbert: {distilbert_result[0]['label']}, Berto: {berto_result[0]['label']}")
    if distilbert_result[0]['label'] == dialogo:
        final_count['distilbert'] = final_count.get('distilbert', 0) + 1
    if berto_result[0]['label'] == dialogo:
        final_count['berto'] = final_count.get('berto', 0) + 1

print(f"""
Final results: 
    Distilbert: {final_count.get('distilbert', 0)}/{len(sample_conflicts)}
    Berto: {final_count.get('berto', 0)}/{len(sample_conflicts)}
""")

True label: HAY_DIALOGO | Distilbert: NO_DIALOGO, Berto: NO_DIALOGO
True label: NO_DIALOGO | Distilbert: NO_DIALOGO, Berto: NO_DIALOGO
True label: HAY_DIALOGO | Distilbert: HAY_DIALOGO, Berto: HAY_DIALOGO
True label: HAY_DIALOGO | Distilbert: NO_DIALOGO, Berto: NO_DIALOGO
True label: NO_DIALOGO | Distilbert: NO_DIALOGO, Berto: NO_DIALOGO
True label: HAY_DIALOGO | Distilbert: NO_DIALOGO, Berto: HAY_DIALOGO

Final results: 
    Distilbert: 3/6
    Berto: 4/6



# Prompt engineering for conflict classification

This section evaluates whether a large language model can perform the dialogue classification task through prompt design alone, without
gradient-based fine-tuning. I compare several prompting strategies, including zero-shot, rubric-based, few-shot, structured-output
prompts and a mixed-prompt, in order to test how much performance depends on instruction quality and response format constraints.

In [32]:
# Conflict level data
print(f"Total conflicts:    {conflict_df.shape[0]}")
print(f"Train:              {train_conflict.shape[0]}")
print(f"Validation:         {val_conflict.shape[0]}")
print(f"Test:               {test_conflict.shape[0]}")

Total conflicts:    246
Train:              172
Validation:         37
Test:               37


In [33]:
# Event level data
print(f"Total events:   {dialogo_df.shape[0]}")
print(f"Train:          {train_df.shape[0]}")
print(f"Validation:     {val_df.shape[0]}")
print(f"Test:           {test_df.shape[0]}")

Total events:   2947
Train:          2030
Validation:     557
Test:           360


In [34]:
def format_case(conflict_text: str, event_text: str) -> str:
    return f"[CONFLICTO]\n{conflict_text}\n\n[EVENTO]\n{event_text}"

def build_single_prompt(
    prompt_style: str,
    conflict_text: str,
    event_text: str,
    few_shot_examples: list[dict[str, str]] | None = None,
) -> dict[str, str]:
    case_text = format_case(conflict_text, event_text)
    system_prompt = (
        "Eres un especialista en conflictos sociales de la Defensoría del Pueblo del Perú."
        "Tu objetivo es clasificar si el evento muestra evidencia de dialogo entre las partes."
        "Las únicas etiquetas válidas son 'HAY_DIALOGO' y 'NO_DIALOGO'."
    )

    if prompt_style == "zero_shot_simple":
        return {
            "prompt_style": prompt_style,
            "response_format": "plain",
            "system_prompt": system_prompt,
            "user_prompt": (
                "Clasifica el siguiente caso. Responde solo con una etiqueta exacta.\n\n"
                f"{case_text}\n\nRespuesta:"
            ),
        }

    if prompt_style == "zero_shot_rubric":
        return {
            "prompt_style": prompt_style,
            "response_format": "plain",
            "system_prompt": system_prompt,
            "user_prompt": (
                "Usa este criterio:\n"
                "- HAY_DIALOGO: hay evidencia de reunión, mesa de diálogo, negociación, mediación, acuerdo o proceso de diálogo activo.\n"
                "- NO_DIALOGO: hay protesta, bloqueo, denuncia, tension o demanda, pero sin evidencia explícita de diálogo.\n"
                "- Si el texto es ambiguo, prioriza la evidencia explícita.\n"
                "Responde solo con una etiqueta exacta.\n\n"
                f"{case_text}\n\nRespuesta:"
            ),
        }

    if prompt_style == "few_shot":
        if not few_shot_examples:
            raise ValueError("few_shot_examples is required for the few_shot prompt")

        examples_text = []
        for idx, example in enumerate(few_shot_examples, start=1):
            examples_text.append(
                (
                    f"Ejemplo {idx}\n"
                    f"{format_case(example['clean_conflict_text'], example['clean_event_text'])}\n\n"
                    f"Etiqueta: {example['label']}"
                )
            )

        return {
            "prompt_style": prompt_style,
            "response_format": "plain",
            "system_prompt": system_prompt,
            "user_prompt": (
                "Aprende de estos ejemplos y luego clasifica el caso final. "
                "Responde solo con una etiqueta exacta.\n\n"
                + "\n\n".join(examples_text)
                + f"\n\nCaso final\n{case_text}\n\nEtiqueta:"
            ),
        }

    if prompt_style == "structured_json":
        return {
            "prompt_style": prompt_style,
            "response_format": "json",
            "system_prompt": system_prompt,
            "user_prompt": (
                'Devuelve solo un JSON valido con esta estructura exacta: {"label":"HAY_DIALOGO o NO_DIALOGO","confidence":0.0,"rationale":"texto breve"}.\n\n'
                f"{case_text}"
            ),
        }
    
    if prompt_style == "structured_few_shot_rubric":
        if not few_shot_examples:
            raise ValueError("few_shot_examples is required for the few_shot prompt")

        examples_text = []
        for idx, example in enumerate(few_shot_examples, start=1):
            examples_text.append(
                (
                    f"Ejemplo {idx}\n"
                    f"{format_case(example['clean_conflict_text'], example['clean_event_text'])}\n\n"
                    f"Etiqueta: {example['label']}"
                )
            )

        return {
            "prompt_style": prompt_style,
            "response_format": "json",
            "system_prompt": system_prompt,
            "user_prompt": (
                "Usa este criterio:\n"
                "- HAY_DIALOGO: hay evidencia de reunión, mesa de diálogo, negociación, mediación, acuerdo o proceso de diálogo activo.\n"
                "- NO_DIALOGO: hay protesta, bloqueo, denuncia, tensión o demanda, pero sin evidencia explícita de diálogo.\n"
                "- Si el texto es ambiguo, prioriza la evidencia explícita.\n\n"

                "Aprende de los siguientes ejemplos y luego clasifica el caso final.\n\n"

                + "\n\n".join(examples_text)

                + f"\n\nCaso final:\n{case_text}\n\n"

                "Devuelve SOLO un JSON válido con esta estructura exacta:\n"
                '{"label":"HAY_DIALOGO o NO_DIALOGO","confidence":0.0,"rationale":"texto breve"}'
            ),
        }

    raise ValueError(f"Unsupported prompt_style: {prompt_style}")


In [35]:
def sample_few_shot_examples(
    train_df: pl.DataFrame,
    label_col: str = "dialogo_label",
    conflict_col: str = "clean_conflict_text",
    event_col: str = "clean_event_text",
    per_label: int = 5,
    seed: int = 42,
) -> list[dict[str, str]]:
    examples: list[dict[str, str]] = []
    rng = np.random.default_rng(seed)

    for label in ('HAY_DIALOGO', 'NO_DIALOGO'):
        label_df = train_df.filter(pl.col(label_col) == label)
        if label_df.height == 0:
            continue
        take = min(per_label, label_df.height)
        indices = rng.choice(label_df.height, size=take, replace=False)
        sampled = label_df[indices]
        for row in sampled.iter_rows(named=True):
            examples.append(
                {
                    "label": row[label_col],
                    "clean_conflict_text": row[conflict_col],
                    "clean_event_text": row[event_col],
                }
            )

    return examples

In [36]:
train_samples = sample_few_shot_examples(train_df, per_label = 10)

In [37]:
test_samples = sample_few_shot_examples(test_df)

In [38]:
zero_shot_prompt = build_single_prompt(
    prompt_style="zero_shot_simple",
    conflict_text=test_samples[0]['clean_conflict_text'],
    event_text=test_samples[0]['clean_event_text'],
)
zero_shot_prompt

{'prompt_style': 'zero_shot_simple',
 'response_format': 'plain',
 'system_prompt': "Eres un especialista en conflictos sociales de la Defensoría del Pueblo del Perú.Tu objetivo es clasificar si el evento muestra evidencia de dialogo entre las partes.Las únicas etiquetas válidas son 'HAY_DIALOGO' y 'NO_DIALOGO'.",
 'user_prompt': 'Clasifica el siguiente caso. Responde solo con una etiqueta exacta.\n\n[CONFLICTO]\nanexo joraoniyoc comunidad campesina san francisco asís yarusyacan demanda empresa nexa resources cumplimiento acuerdo temas empleo local proyectos productivos . adicionalmente , demandan reposición laboral trabajadores empresa minera .\n\n[EVENTO]\n21 junio anexo machcan realizando medida protesta consiste bloqueo vía ingreso instalaciones tajo san gerardo unidad minera antacocha presunta afectación actividades agrícolas comunidad , solicitan reconocimiento pérdidas económicas parte empresa , reposición puestos labores , puntos . 3 julio presente llevó cabo reunión comité pre

In [39]:
zero_shot_rubric_prompt = build_single_prompt(
    prompt_style="zero_shot_rubric",
    conflict_text=test_samples[0]['clean_conflict_text'],
    event_text=test_samples[0]['clean_event_text'],
)
zero_shot_rubric_prompt

{'prompt_style': 'zero_shot_rubric',
 'response_format': 'plain',
 'system_prompt': "Eres un especialista en conflictos sociales de la Defensoría del Pueblo del Perú.Tu objetivo es clasificar si el evento muestra evidencia de dialogo entre las partes.Las únicas etiquetas válidas son 'HAY_DIALOGO' y 'NO_DIALOGO'.",
 'user_prompt': 'Usa este criterio:\n- HAY_DIALOGO: hay evidencia de reunión, mesa de diálogo, negociación, mediación, acuerdo o proceso de diálogo activo.\n- NO_DIALOGO: hay protesta, bloqueo, denuncia, tension o demanda, pero sin evidencia explícita de diálogo.\n- Si el texto es ambiguo, prioriza la evidencia explícita.\nResponde solo con una etiqueta exacta.\n\n[CONFLICTO]\nanexo joraoniyoc comunidad campesina san francisco asís yarusyacan demanda empresa nexa resources cumplimiento acuerdo temas empleo local proyectos productivos . adicionalmente , demandan reposición laboral trabajadores empresa minera .\n\n[EVENTO]\n21 junio anexo machcan realizando medida protesta cons

In [40]:
few_shot_prompt = build_single_prompt(
    prompt_style="few_shot",
    conflict_text=test_samples[0]['clean_conflict_text'],
    event_text=test_samples[0]['clean_event_text'],
    few_shot_examples=train_samples
)
few_shot_prompt

{'prompt_style': 'few_shot',
 'response_format': 'plain',
 'system_prompt': "Eres un especialista en conflictos sociales de la Defensoría del Pueblo del Perú.Tu objetivo es clasificar si el evento muestra evidencia de dialogo entre las partes.Las únicas etiquetas válidas son 'HAY_DIALOGO' y 'NO_DIALOGO'.",
 'user_prompt': "Aprende de estos ejemplos y luego clasifica el caso final. Responde solo con una etiqueta exacta.\n\nEjemplo 1\n[CONFLICTO]\nvecinos asentamiento humano villa hermosa , docentes alumnos i.e . josé pardo barreda alcalde distrital brea negritos piden realice abandono pozos gas petróleo forma técnica ( pasivos ambientales ) , construcción nueva infraestructura i.e . josé pardo barreda afectada afloramiento petróleo constitución reglamento uso aportes empresas petroleras fideicomiso .\n\n[EVENTO]\nsupervisó ie josé pardo barreda , pudo constatar actualmente cuenta servicio vigilancia 24 horas , dividido dos turnos . - asimismo , realizó entrevista encargado empresa l & l

In [41]:
structured_json_prompt = build_single_prompt(
    prompt_style="structured_json",
    conflict_text=test_samples[0]['clean_conflict_text'],
    event_text=test_samples[0]['clean_event_text'],
)
structured_json_prompt

{'prompt_style': 'structured_json',
 'response_format': 'json',
 'system_prompt': "Eres un especialista en conflictos sociales de la Defensoría del Pueblo del Perú.Tu objetivo es clasificar si el evento muestra evidencia de dialogo entre las partes.Las únicas etiquetas válidas son 'HAY_DIALOGO' y 'NO_DIALOGO'.",
 'user_prompt': 'Devuelve solo un JSON valido con esta estructura exacta: {"label":"HAY_DIALOGO o NO_DIALOGO","confidence":0.0,"rationale":"texto breve"}.\n\n[CONFLICTO]\nanexo joraoniyoc comunidad campesina san francisco asís yarusyacan demanda empresa nexa resources cumplimiento acuerdo temas empleo local proyectos productivos . adicionalmente , demandan reposición laboral trabajadores empresa minera .\n\n[EVENTO]\n21 junio anexo machcan realizando medida protesta consiste bloqueo vía ingreso instalaciones tajo san gerardo unidad minera antacocha presunta afectación actividades agrícolas comunidad , solicitan reconocimiento pérdidas económicas parte empresa , reposición puest

In [42]:
build_single_prompt(
    prompt_style="structured_few_shot_rubric",
    conflict_text=test_samples[0]['clean_conflict_text'],
    event_text=test_samples[0]['clean_event_text'],
    few_shot_examples=train_samples
)

{'prompt_style': 'structured_few_shot_rubric',
 'response_format': 'json',
 'system_prompt': "Eres un especialista en conflictos sociales de la Defensoría del Pueblo del Perú.Tu objetivo es clasificar si el evento muestra evidencia de dialogo entre las partes.Las únicas etiquetas válidas son 'HAY_DIALOGO' y 'NO_DIALOGO'.",
 'user_prompt': 'Usa este criterio:\n- HAY_DIALOGO: hay evidencia de reunión, mesa de diálogo, negociación, mediación, acuerdo o proceso de diálogo activo.\n- NO_DIALOGO: hay protesta, bloqueo, denuncia, tensión o demanda, pero sin evidencia explícita de diálogo.\n- Si el texto es ambiguo, prioriza la evidencia explícita.\n\nAprende de los siguientes ejemplos y luego clasifica el caso final.\n\nEjemplo 1\n[CONFLICTO]\nvecinos asentamiento humano villa hermosa , docentes alumnos i.e . josé pardo barreda alcalde distrital brea negritos piden realice abandono pozos gas petróleo forma técnica ( pasivos ambientales ) , construcción nueva infraestructura i.e . josé pardo b

In [43]:
from social_conflicts_peru.utils import parse_json_label, parse_plain_label
from openai import OpenAI
from social_conflicts_peru.config import settings

client = OpenAI(api_key=settings.OPENAI_API_KEY)
MODEL = settings.OPENAI_MODEL

def get_prompt_response(client: OpenAI, model: str, prompt: dict) -> dict[str, str]:

    response = client.responses.create(
        model=model,
        instructions=prompt["system_prompt"],
        input=prompt["user_prompt"],
    )

    raw_text = getattr(response, "output_text", "")

    if prompt["response_format"] == "json":
        label, _, _ = parse_json_label(raw_text)
    else:
        label, _, _ = parse_plain_label(raw_text)

    return  {
            "prompt_style": prompt.get("prompt_style"),
            "model_name": model,
            "raw_response_text": raw_text,
            "normalized_label": label,
            "response_id": getattr(response, "id", None),
        }

In [ ]:
# For testing purposes

# styles = ["zero_shot_simple", "zero_shot_rubric", "few_shot", "structured_json", "structured_few_shot_rubric"]

# results: dict[str, list] = {}

# for style in styles:

#     prompt_responses = []
#     for example in test_samples:

#         prompt = build_single_prompt(
#             prompt_style = style,
#             conflict_text=example['clean_conflict_text'],
#             event_text=example['clean_event_text'],
#             few_shot_examples=train_samples
#         )

#         prompt_responses.append(
#             get_prompt_response(
#                 client = client, 
#                 model = MODEL,
#                 prompt = prompt,
#             )
#         )
    
#     results[style] = prompt_responses

In [ ]:
# For testing purposes
# true_labs = [example['label'] for example in test_samples]
# for style, responses in results.items():
#     labels = [response['normalized_label'] for response in responses]
#     print(f"Results for {style}: {np.mean([lab == true_labs[ix] for ix, lab in enumerate(labels)])}")

Results for zero_shot_simple: 0.8333333333333334
Results for zero_shot_rubric: 1.0
Results for few_shot: 0.6666666666666666
Results for structured_json: 0.8333333333333334
Results for structured_few_shot_rubric: 1.0


# AI Agents to predict conflicts classification

This section extends the prompt-based approach to a simple multi-agent setting. Instead of relying on a single answer, I use two agents
that argue for opposite labels and a third agent that acts as judge. The purpose is to test whether explicit argumentative decomposition
improves classification quality relative to the best single-prompt baseline.

In [44]:
base_prompt = (
    "Eres un especialista en conflictos sociales de la Defensoría del Pueblo del Perú."
    "Tu jefe necesita clasificar si el evento asociado a un conflicto social muestra evidencia de dialogo entre las partes."
    "Las únicas etiquetas válidas son 'HAY_DIALOGO' y 'NO_DIALOGO'."
)

def build_agent_prompts(conflict_text: str, event_text: str) -> dict[str, dict[str, str]]:
    case_text = format_case(conflict_text, event_text)

    return {
        "agent_a": {
            "prompt_style": "agent_a",
            "response_format": "plain",
            "system_prompt": base_prompt + (
                "Tienes que defender la etiqueta 'HAY_DIALOGO' usando solo la evidencia del caso."
            ),
            "user_prompt": (
                f"{case_text}\n\nEscribe un argumento breve de maximo 4 oraciones a favor de 'HAY_DIALOGO'"
            ),
        },
        "agent_b": {
            "prompt_style": "agent_b",
            "response_format": "plain",
            "system_prompt": base_prompt + (
                "Tienes que defender la etiqueta 'NO_DIALOGO' usando solo la evidencia del caso."
            ),
            "user_prompt": (
                f"{case_text}\n\nEscribe un argumento breve de maximo 4 oraciones a favor de 'NO_DIALOGO'"
            ),
        },
    }


In [45]:
def build_judge_prompt(
    conflict_text: str,
    event_text: str,
    agent_a_argument: str,
    agent_b_argument: str,
) -> dict[str, str]:
    case_text = format_case(conflict_text, event_text)
    return {
        "prompt_style": "agent_judge",
        "response_format": "json",
        "system_prompt": (
            "Eres el jefe del departamento de monitoreo de conflictos sociales de la Defensoría del Pueblo del Perú."
            "Tu objetivo es clasificar si el evento asociado a un conflicto social muestra evidencia de dialogo entre las partes."
            "Debes decidir entre 'HAY_DIALOGO' y 'NO_DIALOGO' usando solo la evidencia del caso y los argumentos provistos por tus especialistas"
        ),
        "user_prompt": (
            f"{case_text}\n\n"
            f"Argumento a favor de 'HAY_DIALOGO':\n{agent_a_argument}\n\n"
            f"Argumento a favor de 'NO_DIALOGO':\n{agent_b_argument}\n\n"
            'Devuelve solo un JSON valido con: {"label":"HAY_DIALOGO o NO_DIALOGO","winning_side":"A o B","rationale":"texto breve"}.'
        ),
    }

In [46]:
def get_agents_reponses(client: OpenAI, model: str, test_example: dict) -> dict[str, dict[str, str]]:
    
    agents_prompts = build_agent_prompts(
        conflict_text=test_example['clean_conflict_text'],
        event_text=test_example['clean_event_text'],
    )

    agents_responses : dict[str, dict] = {}

    for agent, instructions in agents_prompts.items():
        response = get_prompt_response(
            client = client,
            model = model,
            prompt = instructions
        )
    
        agents_responses[agent] = {
            "prompt_style": response.get("prompt_style"),
            "response": response.get("raw_response_text"),
        }
    
    return agents_responses

In [47]:
agents_responses = get_agents_reponses(client, MODEL, test_samples[0])
agents_responses

{'agent_a': {'prompt_style': 'agent_a',
  'response': 'El 3 de julio se llevó a cabo una reunión del comité de prevención de conflictos sociales en Pasco con la participación de Anexo Machan y la empresa Nexa, lo que constituye un espacio de diálogo formal entre las partes. En esa reunión las partes establecieron acuerdos sobre la medida de protesta, demostrando que hubo negociación y acuerdos mutuos. La realización de la reunión y la existencia de acuerdos son evidencia directa de que sí hubo diálogo entre la comunidad y la empresa.'},
 'agent_b': {'prompt_style': 'agent_b',
  'response': 'Aunque hubo una reunión el 3 de julio con participación de anexo Machan y la empresa Nexa, las partes acordaron mantener la medida de protesta en lugar de resolver las demandas. Persistieron el bloqueo y los enfrentamientos entre agentes de seguridad de la minera y manifestantes, lo que evidencia escalamiento y no un diálogo constructivo. Las demandas de reposición laboral y reconocimiento de pérdid

In [48]:
judge_prompt = build_judge_prompt(
    conflict_text=test_samples[0]['clean_conflict_text'],
    event_text=test_samples[0]['clean_event_text'],
    agent_a_argument=agents_responses['agent_a']['response'],
    agent_b_argument=agents_responses['agent_b']['response']
)

judge_prompt

{'prompt_style': 'agent_judge',
 'response_format': 'json',
 'system_prompt': "Eres el jefe del departamento de monitoreo de conflictos sociales de la Defensoría del Pueblo del Perú.Tu objetivo es clasificar si el evento asociado a un conflicto social muestra evidencia de dialogo entre las partes.Debes decidir entre 'HAY_DIALOGO' y 'NO_DIALOGO' usando solo la evidencia del caso y los argumentos provistos por tus especialistas",
 'user_prompt': '[CONFLICTO]\nanexo joraoniyoc comunidad campesina san francisco asís yarusyacan demanda empresa nexa resources cumplimiento acuerdo temas empleo local proyectos productivos . adicionalmente , demandan reposición laboral trabajadores empresa minera .\n\n[EVENTO]\n21 junio anexo machcan realizando medida protesta consiste bloqueo vía ingreso instalaciones tajo san gerardo unidad minera antacocha presunta afectación actividades agrícolas comunidad , solicitan reconocimiento pérdidas económicas parte empresa , reposición puestos labores , puntos . 3

In [58]:
def get_judge_response(client: OpenAI, model: str, test_example: dict, agents_responses: dict) -> dict[str, str]:
    
    judge_prompt = build_judge_prompt(
        conflict_text=test_example['clean_conflict_text'],
        event_text=test_example['clean_event_text'],
        agent_a_argument=agents_responses['agent_a']['response'],
        agent_b_argument=agents_responses['agent_b']['response']
    )

    final_response = get_prompt_response(
            client = client,
            model = model,
            prompt = judge_prompt
        )
    
    return final_response

In [59]:
def get_multi_agent_response(client: OpenAI, model: str, test_example: dict) -> tuple[dict[str, str], dict[str, dict[str, str]]]:

    agents_responses = get_agents_reponses(client, model, test_example)

    return get_judge_response(
        client = client, 
        model = model, 
        test_example = test_example, 
        agents_responses = agents_responses
    ), agents_responses

In [ ]:
# For testing purposes

# multi_agent_responses = [
#     get_multi_agent_response(
#         client = client,
#         model = MODEL,
#         test_example = example
#     )
#     for example in test_samples]
# multi_agent_responses

['{"label":"HAY_DIALOGO","winning_side":"A","rationale":"Existe un acta suscrita entre la comunidad y las autoridades municipales con compromisos concretos y plazo (31 de octubre), lo que constituye evidencia de diálogo entre las partes."}',
 '{"label":"HAY_DIALOGO","winning_side":"A","rationale":"El 13/10 hubo una reunión entre autoridades, fiscalía, oficina defensorial, PCM, PNP, capitanía y representantes de armadores que acordaron ingreso a la zona (fecha tentativa 14/10), y el 14/10 la ANA remitió informe técnico a la oficina defensorial; estos hechos muestran coordinación e intercambio de información entre las partes presentes."}',
 '{"label":"NO_DIALOGO","winning_side":"B","rationale":"La reunión fue entre la Defensoría y la Dirección de Titulación de Tierras/Catastro, no entre las comunidades en conflicto; solo se acordó solicitar más información, sin evidencia de mesas de diálogo o comunicación directa entre Cochas y San Francisco Macón."}',
 '{"label":"NO_DIALOGO","winning_si

In [ ]:
# For testing purposes

# true_labs = [example['label'] for example in test_samples]

# labels = [json.loads(response)['label'] for response in multi_agent_responses]

# print(f"Results for multiagent classification: {np.mean([lab == true_labs[ix] for ix, lab in enumerate(labels)])}")

Results for multiagent classification: 0.8333333333333334


# Final Review: Prompt Engineering + Multi-Agent

This final section consolidates the evaluation of the prompt-engineering and multi-agent approaches under a common review framework.
Using the same frozen evaluation slice, I compare predictive performance, invalid-format rate, latency, token usage, and estimated cost.
This allows me to assess not only which approach is more accurate, but also which one is more robust and efficient in practice.

In [82]:
from social_conflicts_peru.utils import (
    summarize_prediction_df,
    build_confusion_df,
)

EVAL_SEED = 42
TARGET_PER_CLASS = min(
    25,
    test_df.filter(pl.col("dialogo_label") == "HAY_DIALOGO").height,
    test_df.filter(pl.col("dialogo_label") == "NO_DIALOGO").height,
)

REVIEW_OUTPUT_DIR = directories.ROOT_DIR / "nlp_analysis" / "llm_review"
REVIEW_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

review_prompt_styles = [
    "zero_shot_simple",
    "zero_shot_rubric",
    "few_shot",
    "structured_json",
    "structured_few_shot_rubric",
]

hay_df = test_df.filter(pl.col("dialogo_label") == "HAY_DIALOGO").sample(
    n=TARGET_PER_CLASS, seed=EVAL_SEED
)

no_df = test_df.filter(pl.col("dialogo_label") == "NO_DIALOGO").sample(
    n=TARGET_PER_CLASS, seed=EVAL_SEED
)

eval_df = (
    pl.concat([hay_df, no_df])
    .sample(fraction=1.0, seed=EVAL_SEED)
    .with_row_index(name="row_id")
)

few_shot_examples = sample_few_shot_examples(train_df, per_label=5, seed=EVAL_SEED)
few_shot_examples_df = pl.DataFrame(few_shot_examples)
print(f"Review rows: {eval_df.height}")
print(eval_df["dialogo_label"].value_counts().sort("dialogo_label"))

Review rows: 50
shape: (2, 2)
┌───────────────┬───────┐
│ dialogo_label ┆ count │
│ ---           ┆ ---   │
│ str           ┆ u32   │
╞═══════════════╪═══════╡
│ HAY_DIALOGO   ┆ 25    │
│ NO_DIALOGO    ┆ 25    │
└───────────────┴───────┘


In [83]:
def run_prompt_review(eval_df: pl.DataFrame, prompt_styles: list[str]):
    rows = []

    for example in eval_df.iter_rows(named=True):
        for style in prompt_styles:
            prompt = build_single_prompt(
                prompt_style=style,
                conflict_text=example["clean_conflict_text"],
                event_text=example["clean_event_text"],
                few_shot_examples=few_shot_examples,
            )
            result = get_prompt_response(
                client = client, 
                model = MODEL,
                prompt = prompt,
            )
            rows.append({
                "row_id": example["row_id"],
                "true_label": example["dialogo_label"],
                "conflict_uid": example["conflict_uid"],
                "prompt_style": style,
                "model_name": result["model_name"],
                "raw_response_text": result["raw_response_text"],
                "normalized_label": result["normalized_label"],
                "response_id": result["response_id"],
                "is_correct": result["normalized_label"] == example["dialogo_label"],
            })

    return pl.DataFrame(rows)

In [84]:
def run_multi_agent_review(eval_df: pl.DataFrame):
    rows = []

    for example in eval_df.iter_rows(named=True):

        multi_agent_response, agent_outputs = get_multi_agent_response(
            client = client,
            model = MODEL,
            test_example = example
        )
        
        rows.append({
            "row_id": example["row_id"],
            "true_label": example["dialogo_label"],
            "conflict_uid": example["conflict_uid"],
            "prompt_style": "multi_agent",
            "model_name": MODEL,
            "normalized_label": multi_agent_response["normalized_label"],
            "response_id": multi_agent_response["response_id"],
            "raw_response_text": multi_agent_response["raw_response_text"],
            "agent_a_argument": agent_outputs["agent_a"]["response"],
            "agent_b_argument": agent_outputs["agent_b"]["response"],
            "is_correct": multi_agent_response["normalized_label"] == example["dialogo_label"],
        })

    return pl.DataFrame(rows)

In [ ]:
# prompt_review_raw_df = run_prompt_review(eval_df, review_prompt_styles)
# prompt_review_raw_df.write_csv(REVIEW_OUTPUT_DIR / "prompt_styles_raw.csv")
prompt_review_raw_df = pl.read_csv(REVIEW_OUTPUT_DIR / "prompt_styles_raw.csv")
prompt_review_raw_df.head()

row_id,true_label,conflict_uid,prompt_style,model_name,raw_response_text,normalized_label,response_id,is_correct
i64,str,str,str,str,str,str,str,bool
0,"""HAY_DIALOGO""","""CF-0000351""","""zero_shot_simple""","""gpt-5-mini""","""HAY_DIALOGO""","""HAY_DIALOGO""","""resp_05df4e7240b0c0770069b0d50…",true
0,"""HAY_DIALOGO""","""CF-0000351""","""zero_shot_rubric""","""gpt-5-mini""","""HAY_DIALOGO""","""HAY_DIALOGO""","""resp_0b7b2c9e68b100400069b0d51…",true
0,"""HAY_DIALOGO""","""CF-0000351""","""few_shot""","""gpt-5-mini""","""HAY_DIALOGO""","""HAY_DIALOGO""","""resp_0df3d8cc65a426360069b0d51…",true
0,"""HAY_DIALOGO""","""CF-0000351""","""structured_json""","""gpt-5-mini""","""{""label"":""HAY_DIALOGO"",""confid…","""HAY_DIALOGO""","""resp_011c3db0f1fc3db40069b0d51…",true
0,"""HAY_DIALOGO""","""CF-0000351""","""structured_few_shot_rubric""","""gpt-5-mini""","""{""label"":""HAY_DIALOGO"",""confid…","""HAY_DIALOGO""","""resp_05338d5a730eefc50069b0d52…",true


In [86]:
multi_agent_raw_df = run_multi_agent_review(eval_df)
multi_agent_raw_df.write_csv(REVIEW_OUTPUT_DIR / "multi_agent_raw.csv")
multi_agent_raw_df = pl.read_csv(REVIEW_OUTPUT_DIR / "multi_agent_raw.csv")
multi_agent_raw_df.head()

row_id,true_label,conflict_uid,prompt_style,model_name,normalized_label,response_id,raw_response_text,agent_a_argument,agent_b_argument,is_correct
i64,str,str,str,str,str,str,str,str,str,bool
0,"""HAY_DIALOGO""","""CF-0000351""","""multi_agent""","""gpt-5-mini""","""HAY_DIALOGO""","""resp_03f593e4ff08dbce0069b0d98…","""{""label"":""HAY_DIALOGO"",""winnin…","""HAY_DIALOGO: en entrevistas la…","""En las entrevistas solo se inf…",true
1,"""HAY_DIALOGO""","""CF-0000311""","""multi_agent""","""gpt-5-mini""","""HAY_DIALOGO""","""resp_0e35e6850f231faa0069b0d9a…","""{""label"":""HAY_DIALOGO"",""winnin…","""HAY_DIALOGO. Se realizaron reu…","""Aunque se realizaron reuniones…",true
2,"""HAY_DIALOGO""","""CF-0000041""","""multi_agent""","""gpt-5-mini""","""HAY_DIALOGO""","""resp_0779f451c33369290069b0d9c…","""{""label"":""HAY_DIALOGO"",""winnin…","""El Frente único envió el 6 de …","""No hay evidencia de diálogo ef…",true
3,"""HAY_DIALOGO""","""CF-0000113""","""multi_agent""","""gpt-5-mini""","""NO_DIALOGO""","""resp_05558ca488f5f6370069b0d9d…","""{""label"":""NO_DIALOGO"",""winning…","""Los pobladores exigieron que l…","""NO_DIALOGO. El evento describe…",false
4,"""HAY_DIALOGO""","""CF-0000061""","""multi_agent""","""gpt-5-mini""","""NO_DIALOGO""","""resp_075c28e0d84ca87e0069b0d9f…","""{""label"":""NO_DIALOGO"",""winning…","""La comunidad campesina Sulcán …","""No hay indicios en el caso de …",false


In [89]:
agent_vs_single_prompt_df = pl.concat(
    [prompt_review_raw_df, multi_agent_raw_df],
    how="diagonal_relaxed",
)

In [95]:
models_summary_df = summarize_prediction_df(
    agent_vs_single_prompt_df,
    ["prompt_style", "model_name"],
)

models_summary_df = models_summary_df.sort(
    ["f1_macro", "balanced_accuracy", "accuracy"],
    descending=[True, True, True],
)
models_summary_df[["prompt_style", "f1_macro", "f1_hay_dialogo", "f1_no_dialogo"]]

prompt_style,f1_macro,f1_hay_dialogo,f1_no_dialogo
str,f64,f64,f64
"""multi_agent""",0.739896,0.745098,0.734694
"""zero_shot_simple""",0.715909,0.75,0.681818
"""few_shot""",0.698916,0.716981,0.680851
"""structured_json""",0.69697,0.727273,0.666667
"""structured_few_shot_rubric""",0.653203,0.701754,0.604651
"""zero_shot_rubric""",0.653203,0.701754,0.604651


### Final Results

The results indicate that multi_agent achieved the best overall performance, with the highest f1_macro (0.740) and the most balanced
results across both classes (0.745 for HAY_DIALOGO and 0.735 for NO_DIALOGO). This suggests that the agent-based setup not only improves
aggregate performance but also reduces the gap between the two labels.

Among the single-prompt methods, zero_shot_simple performed best, with f1_macro = 0.716. Its performance was stronger for HAY_DIALOGO
(0.750) than for NO_DIALOGO (0.682), but it still showed relatively balanced behavior compared with the other prompt-engineering variants.
few_shot and structured_json followed closely behind, although both were somewhat weaker on the NO_DIALOGO class.

Finally, structured_few_shot_rubric and zero_shot_rubric produced the lowest scores, with identical f1_macro values (0.653). In both
cases, performance on NO_DIALOGO remained the weakest part of the model, indicating that adding rubric-style instructions did not improve
class discrimination in this experiment. Overall, the table suggests that the multi-agent design was the most effective approach, while
simpler prompting strategies outperformed the more heavily structured rubric-based prompts.